In [1]:
import sys
sys.path.append("../utils")

from coord import *
from hubs_coordpt2 import *
import pandas as pd
import re

def extract_state(hub_name):
    """Extrae el primer código de estado de un hub: 'Dallas (TX)' → 'TX', 'Brooklyn (NJ, NY)' → 'NJ'."""
    m = re.search(r'\(([A-Z]{2})', str(hub_name))
    return m.group(1) if m else None

/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Identificación de Hubs Origen

In [2]:
df = pd.read_parquet('parquetloadsfull.parquet')
df = df.reset_index(drop=True)
df.loc[df['Posted'].isna(), 'Posted'] = df.loc[df['Posted'].isna(), 'PickUp']
df.shape

(57573489, 20)

In [3]:
state_codes = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE',
    'FLORIDA': 'FL', 'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID',
    'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS',
    'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS',
    'MISSOURI': 'MO', 'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV',
    'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH', 'OKLAHOMA': 'OK',
    'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT',
    'VERMONT': 'VT', 'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY'
}

df['StateOrigin'] = df['StateOrigin'].replace(state_codes)
df['StateDestination'] = df['StateDestination'].replace(state_codes)

In [4]:
hubsorigin = (
    df.loc[
        (df['HubOrigin'].isna()) & (df['LatOrigin'].notna()) & (df['LngOrigin'].notna()),
        ['LatOrigin', 'LngOrigin', 'CityOrigin', 'StateOrigin', 'ID']
    ].rename(columns={'CityOrigin': 'city', 'StateOrigin': 'state', 'LatOrigin': 'lat', 'LngOrigin': 'lng'})
)
print(hubsorigin.shape)
hubsorigin.head()

(3784319, 5)


,lat,lng,city,state,ID
1873251,33.893398,-117.28230,MARCH AIR FORCE BASE,NaN,493002876
1879421,33.300970,-87.04774,MC CALLA,NaN,494468867
1882478,46.505590,-116.55082,LENORE,NaN,494976632
1883821,45.273250,-122.23474,CURRINSVILLE,NaN,492596406
1888366,32.886100,-96.43884,ROCKWELL,NaN,489224233


In [5]:
pt1 = hubsorigin.drop_duplicates(subset=['lat', 'lng'])
pt1.shape

(85867, 5)

In [6]:
hub_assign = pd.read_csv('../utils/hubs_assign.csv')
hubs_1 = hub_assignment(pt1, hub_assign, '../utils/hubs_id.xlsx')

Cantidad de ciudades inicial: 85867
Cantidad de ciudades a procesar: 85867
Cantidad de ciudades a procesar borrando duplicados: 85867


,city,state_id,lat,lng
0,MARCH AIR FORCE BASE,NaN,33.893398,-117.28230
1,MC CALLA,NaN,33.300970,-87.04774
2,LENORE,NaN,46.505590,-116.55082
3,CURRINSVILLE,NaN,45.273250,-122.23474
4,ROCKWELL,NaN,32.886100,-96.43884


voy a procesar estas ciudades 66213
se guardaron los cambios2


,lat,lng,city,state,ID,new_name,hubs_50m,hubs_100m,hubs_150m
0,33.893398,-117.28230,MARCH AIR FORCE BASE,NaN,493002876,San Diego (CA),[131],"[127, 131]","[125, 127, 131]"
1,33.300970,-87.04774,MC CALLA,NaN,494468867,Birmingham (AL),"['1', '6']","['1', '2', '4', '6']","['1', '2', '3', '4', '5', '6', '19', '35', '39']"
2,46.505590,-116.55082,LENORE,NaN,494976632,Winchester (ID),[102],"[100, 102]","[100, 102, 134, 153, 158]"
3,45.273250,-122.23474,CURRINSVILLE,NaN,492596406,Hood River (OR),"[154, 160]","[152, 154, 160]","[152, 154, 156, 159, 160]"
4,32.886100,-96.43884,ROCKWELL,NaN,489224233,Dallas (TX),[185],"[107, 185, 188, 193]","[107, 185, 188, 193]"


In [7]:
# Rellenar estado faltante en hubs_1 usando el código del hub
mask = hubs_1['state'].isna()
hubs_1.loc[mask, 'state'] = hubs_1.loc[mask, 'near_hub'].apply(extract_state)

In [8]:
hubs_1.to_csv('../otros/hubsdelfullparquet.csv', index=False, float_format='%.10f')

In [9]:
hubs_1 = pd.read_csv('../otros/hubsdelfullparquet.csv')

In [10]:
hubsorigin = hubsorigin.merge(hubs_1[['lat', 'lng', 'near_hub']], on=['lat', 'lng'], how='left')

In [11]:
pt2 = hubsorigin.loc[hubsorigin['near_hub'].isna(), ['lat', 'lng', 'city', 'state']]
pt2 = pt2.drop_duplicates(subset=['lat', 'lng'])
print(f"Coordenadas de origen sin hub tras primera pasada: {pt2.shape[0]}")
pt2.shape

Coordenadas de origen sin hub tras primera pasada: 5866


(5866, 4)

In [12]:
hub_assign = pd.read_csv('../utils/hubs_assign.csv')
hubs_2 = hub_assignment(pt2, hub_assign, '../utils/hubs_id.xlsx')

Cantidad de ciudades inicial: 5866
Cantidad de ciudades a procesar: 5866
Cantidad de ciudades a procesar borrando duplicados: 5866


,city,state_id,lat,lng
0,INTERNATIONAL PAPER VICKSBURG MILL,NaN,32.528514,-90.775099
1,HAZLE TOWNSHIP,NaN,40.956471,-76.056173
2,"NOTTINGHAM,CH",NaN,39.750103,-76.014076
3,WOONSOCKET,SD,44.051170,-98.275360
4,ENKA,NC,35.539720,-82.640040


voy a procesar estas ciudades 5455
se guardaron los cambios2


,lat,lng,city,state,new_name,hubs_50m,hubs_100m,hubs_150m
0,32.528514,-90.775099,INTERNATIONAL PAPER VICKSBURG MILL,NaN,Jackson (MS),[38],"[36, 38, 58]","[18, 36, 37, 38, 39, 56, 58]"
1,40.956471,-76.056173,HAZLE TOWNSHIP,NaN,Allentown (PA),['170'],"['166', '170', '171']","['71', '166', '167', '168', '169', '170', '171']"
2,39.750103,-76.014076,"NOTTINGHAM,CH",NaN,Philadelphia (PA),"[71, 166]","[71, 166, 170, 171]","[49, 71, 112, 166, 169, 170, 171]"
3,44.051170,-98.275360,WOONSOCKET,SD,Sioux Falls (SD),[],"[62, 64, 65]","[62, 63, 64, 65, 139]"
4,35.539720,-82.640040,ENKA,NC,Greenville (SC),['176'],"['141', '176', '179']","['21', '46', '140', '141', '176', '179', '180']"


In [13]:
hubs_2.to_csv('../otros/hubsdelfullparquet2.csv', index=False, float_format='%.10f')

In [14]:
hubs_2 = pd.read_csv('../otros/hubsdelfullparquet2.csv')
hubs_origenes = (
    pd.concat([hubs_1, hubs_2])
    .drop_duplicates(subset=['lat', 'lng'])
    .reset_index(drop=True)
)
# Acá están todas las coordenadas de origen procesadas

In [15]:
hubsorigin = (
    hubsorigin
    .merge(hubs_origenes[['lat', 'lng', 'near_hub']], on=['lat', 'lng'], how='left')
    .drop(columns='near_hub_x')
    .rename(columns={'near_hub_y': 'near_hub'})
)

In [16]:
# Rellenar estado faltante en hubsorigin
mask = hubsorigin['state'].isna()
hubsorigin.loc[mask, 'state'] = hubsorigin.loc[mask, 'near_hub'].apply(extract_state)

In [17]:
df = df.merge(
    hubsorigin[['ID', 'near_hub', 'state']].drop_duplicates(subset='ID'),
    on='ID',
    how='left'
)

In [18]:
mask_hub = (df['HubOrigin'].isna()) & (df['LatOrigin'].notna()) & (df['LngOrigin'].notna())
df.loc[mask_hub, 'HubOrigin'] = df.loc[mask_hub, 'near_hub']

In [19]:
mask_state = (df['StateOrigin'].isna()) & (df['state'].notna())
df.loc[mask_state, 'StateOrigin'] = df.loc[mask_state, 'state']
df = df.drop(['state', 'near_hub'], axis=1)

In [20]:
# --- Verificación de origen ---
sin_hub_con_coords = df[(df['HubOrigin'].isna()) & (df['LatOrigin'].notna()) & (df['LngOrigin'].notna())]
hub_sin_estado = df[(df['HubOrigin'].notna()) & (df['StateOrigin'].isna())]
print(f"Orígenes con coords pero sin hub : {len(sin_hub_con_coords):,}")
print(f"Orígenes con hub pero sin estado  : {len(hub_sin_estado):,}")

Orígenes con coords pero sin hub : 306,028
Orígenes con hub pero sin estado  : 20,026


In [21]:
df.shape

(57573489, 20)

In [22]:
# df.to_parquet('parquetloadsfull.parquet')

In [23]:
hubs_name = pd.read_excel('../utils/hubs_id.xlsx')
old_hubs = hubs_name['old_name']
new_hubs = hubs_name['new_name']
hubschange = set(old_hubs).difference(set(new_hubs))

In [24]:
city_pairs = {
    "Opelika (AL)": "Auburn (AL)", "Lake Havasu City (AZ)": "Quartzsite (AZ)",
    "Flagstaff (AZ)": "Cottonwood (AZ)", "Phoenix (AZ)": "Glendale (AZ)",
    "Springdale (AR)": "Fayetteville (AR)", "Plumerville (AR)": "Little Rock (AR)",
    "Paragould (AR)": "Jonesboro (AR)", "Glenwood (AR)": "Texarkana (AR)",
    "Hazlehurst (GA)": "Savannah (GA)", "Shady Dale (GA)": "Athens (GA)",
    "Louisville (GA)": "Milledgeville (GA)", "Cordele (GA)": "Albany (GA)",
    "Sullivan (IL)": "Effingham (IL)", "Peru (IL)": "Peoria (IL)",
    "Council Bluffs (IA)": "Omaha (NE)", "Davenport (IA)": "Iowa city (IA)",
    "Ripley (MS)": "Tupelo (MS)", "Lumberton (MS)": "Hattiesburg (MS)",
    "Vicksburg (MS)": "Jackson (MS)", "Meridian (MS)": "Philadelphia (MS)",
    "Milford (UT)": "Circleville (UT)", "Clearfield (UT)": "Ogden (UT)",
    "Mount Crawford (VA)": "Waynesboro (VA)", "Martinsville (VA)": "Lynchburg (VA)",
    "Glade Spring (VA)": "Wytheville (VA)", "Winchester (VA)": "Warrenton (VA)",
    "Kansas City (KS)": "Lawrence (KS)", "Salina (KS)": "Junction city (KS)",
    "Zwolle (LA)": "Alexandria (LA)", "Simsboro (LA)": "West Monroe (LA)",
    "Harrold (SD)": "Pierre (SD)", "Bridger (SD)": "Midland (SD)",
    "Brookings (SD)": "Watertown (SD)", "Rapid City (SD)": "Sturgis (SD)",
    "Clarkson (KY)": "Hodgenville (KY)", "Frankfort (KY)": "Carrollton (KY)",
    "Bowling Green (KY)": "Central City (KY)", "Nicholasville (KY)": "Lexington (KY)",
    "Detroit (MI)": "Ann Arbor (MI)", "Bridgeport (MI)": "Saginaw (MI)",
    "Grayling (MI)": "Gaylord (MI)", "Evart (MI)": "Reed City (MI)",
    "Harrisonville (MO)": "Lexington (MO)", "Memphis (MO)": "Kirksville (MO)",
    "Columbia (MO)": "Jefferson City (MO)", "Lebanon (MO)": "Springfield (MO)",
    "Fredericktown (MO)": "Poplar Bluff (MO)", "Sandborn (IN)": "Bedford (IN)",
    "South Bend (IN)": "Warsaw (IN)", "Portland (IN)": "Muncie (IN)",
    "Brandon (MN)": "Alexandria (MN)", "Bemidji (MN)": "Park Rapids (MN)",
    "Idaho Falls (ID)": "Pocatello (ID)", "Priest River (ID)": "Coeur d'alene (ID)",
    "Jerome (ID)": "Twin Falls (ID)", "Kooskia (ID)": "Winchester (ID)",
    "Cincinnati (OH)": "Dayton (OH)", "Lima (OH)": "Findlay (OH)",
    "Cleveland (OH)": "Akron (OH)", "Antlers (OK)": "Durant (OK)",
    "Huntington (WV)": "Charleston (WV)", "Elkins (WV)": "Fairmont (WV)",
    "Ranson (WV)": "Cumberland (MD)", "Spencer (WI)": "Schofield (WI)",
    "Delta (CO)": "Ridgway (CO)", "Olney Springs (CO)": "Pueblo (CO)",
    "Green River (WY)": "Rock Springs (WY)", "Anderson (CA)": "Red Bluff (CA)",
    "Superior (MT)": "Missoula (MT)", "Sutton (NE)": "Grand Island (NE)",
    "Ogallala (NE)": "North Platte (NE)", "Omaha (NE)": "Lincoln (NE)",
    "Wayne (NE)": "Sioux City (IA)", "Athens (TN)": "Crossville (TN)",
    "Morristown (TN)": "Knoxville (TN)", "Fayetteville (TN)": "Chattanooga (TN)",
    "Jackson (TN)": "Brownsville (TN)", "Live Oak (FL)": "Tallahassee (FL)",
    "Eugene (OR)": "Salem (OR)", "Richland (OR)": "Baker City (OR)",
    "Carson (WA, OR)": "Hood River (OR)", "Riddle (OR)": "Grants Pass (OR)",
    "Seattle (WA)": "Tacoma (WA)", "Sumas (WA)": "Mount Vernon (WA)",
    "Moses Lake (WA)": "Odessa (WA)", "Pasco (OR, WA)": "Yakima (WA)",
    "Troutdale (OR, WA)": "St. Helens (OR)",
    "Island Pond (VT, NH)": "St. Johnsbury (VT)",
    "New Bedford (RI, MA, CT)": "Boston (MA)",
    "Westfield (CT, MA, VT)": "Springfield (MA)",
    "Newark (NJ, PA, DE)": "Philadelphia (PA)",
    "Buffalo (NY, PA)": "Buffalo (NY)", "Syracuse (NY, PA)": "Ithaca (NY)",
    "Brooklyn (NJ, NY)": "New York (NY)", "Hazleton (PA, NJ)": "Allentown (PA)",
    "York (PA)": "Carlisle (PA)", "Johnstown (NY)": "Albany (NY)",
    "Zelienople (PA)": "Pittsburgh (PA)",
    'Summerville (SC)': 'Orangeburg (SC)', 'Florence (NC, SC)': 'Florence (SC)',
    'Gaffney (SC, NC)': 'Greenville (SC)', 'New Bern (NC)': 'Kinston (NC)',
    'Charlotte (NC, SC)': 'Charlotte (NC)', 'Greensboro (NC)': 'Winston-Salem (NC)',
    'El Paso (NM, TX)': 'El Paso (TX)', "Pharr (TX)": 'Mcallen (TX)',
    "Jasper (TX)": "Lufkin (TX)", "Muleshoe (NM, TX)": "Lubbock (TX)",
}

In [25]:
df.loc[df['HubOrigin'].isin(hubschange), 'HubOrigin'] = df['HubOrigin'].replace(city_pairs)

In [26]:
df.loc[~df['HubOrigin'].isin(new_hubs), 'HubOrigin'] = df['HubOrigin'].replace(
    {'Manchester (MA, NH, ME)': 'Manchester (NH)'}
)

In [27]:
# Rellenar estado desde hub para cualquier fila que tenga hub pero no estado (incluye hubs pre-existentes)
mask = (df['StateOrigin'].isna()) & (df['HubOrigin'].notna())
df.loc[mask, 'StateOrigin'] = df.loc[mask, 'HubOrigin'].apply(extract_state)

In [28]:
df.shape

(57573489, 20)

In [29]:
# df.to_parquet('parquetloadsfull.parquet')

#### Identificación de Hubs Destinos

In [30]:
import pandas as pd
df = pd.read_parquet('parquetloadsfull.parquet')

In [31]:
hubsdestination = (
    df.loc[
        (df['HubDestination'].isna()) & (df['LatDestination'].notna()) & (df['LngDestination'].notna()),
        ['LatDestination', 'LngDestination', 'CityDestination', 'StateDestination', 'ID']
    ].rename(columns={
        'CityDestination': 'city', 'StateDestination': 'state',
        'LatDestination': 'lat', 'LngDestination': 'lng'
    })
)
hubsdestination.shape

(3790357, 5)

In [32]:
test  = pd.read_csv('../otros/hubsdelfullparquet.csv')
test2 = pd.read_csv('../otros/hubsdelfullparquet2.csv')

In [33]:
hubs_origenes = pd.concat([test, test2]).drop_duplicates(subset=['lat', 'lng'])

In [34]:
# Aprovechar coordenadas ya resueltas en origen
hubsdestination = hubsdestination.merge(
    hubs_origenes[['lat', 'lng', 'near_hub']], on=['lat', 'lng'], how='left'
)

In [35]:
hubsdestination.shape

(3790357, 6)

In [36]:
# Coordenadas que aún no tienen hub asignado (no estaban en los csvs de origen)
pt1 = (
    hubsdestination.loc[hubsdestination['near_hub'].isna()]
    .drop('near_hub', axis=1)
    .drop_duplicates(subset=['lat', 'lng'])
)
print(f"Destinos sin hub tras reutilizar coords de origen: {pt1.shape[0]}")
pt1.shape

Destinos sin hub tras reutilizar coords de origen: 79184


(79184, 5)

In [37]:
hub_assign = pd.read_csv('../utils/hubs_assign.csv')
hubs_1 = hub_assignment(pt1, hub_assign, '../utils/hubs_id.xlsx')

Cantidad de ciudades inicial: 79184
Cantidad de ciudades a procesar: 79184
Cantidad de ciudades a procesar borrando duplicados: 79184


,city,state_id,lat,lng
0,SPRINGFIELD TOWNSHIP,NaN,40.686070,-74.322300
1,O'NEALS,NaN,37.138401,-119.685300
2,LAKE VIEW TERRACE,NaN,34.284401,-118.366890
3,SCOTT,NaN,32.344398,-89.327102
4,FORT JACKSON,NaN,34.029400,-80.774280


voy a procesar estas ciudades 69117
se guardaron los cambios2


,lat,lng,city,state,ID,new_name,hubs_50m,hubs_100m,hubs_150m
0,40.686070,-74.322300,SPRINGFIELD TOWNSHIP,NaN,494411154,New York (NY),[169],"[166, 169, 170]","[165, 166, 169, 170]"
1,37.138401,-119.685300,O'NEALS,NaN,495610380,Fresno (CA),[129],"[128, 129]","[125, 128, 129, 130]"
2,34.284401,-118.366890,LAKE VIEW TERRACE,NaN,495598763,Los Angeles (CA),[127],"[125, 127, 131]","[125, 127, 131]"
3,32.344398,-89.327102,SCOTT,NaN,495399133,Philadelphia (MS),"[38, 39]","[1, 36, 37, 38, 39]","[1, 5, 35, 36, 37, 38, 39]"
4,34.029400,-80.774280,FORT JACKSON,NaN,495561984,Orangeburg (SC),[174],"[174, 175, 176, 179]","[23, 174, 175, 176, 179, 180]"


In [38]:
hubs_1.to_csv('../otros/hubsdestinations.csv', index=False, float_format='%.10f')

In [39]:
# Rellenar estado faltante en hubs_1 usando el código del hub
mask = hubs_1['state'].isna()
hubs_1.loc[mask, 'state'] = hubs_1.loc[mask, 'near_hub'].apply(extract_state)

In [40]:
hubsdestination = hubsdestination.merge(
    hubs_1[['lat', 'lng', 'near_hub']], on=['lat', 'lng'], how='left'
)
hubsdestination.loc[hubsdestination['near_hub_x'].isna(), 'near_hub_x'] = hubsdestination['near_hub_y']
hubsdestination = hubsdestination.drop('near_hub_y', axis=1).rename(columns={'near_hub_x': 'near_hub'})

In [41]:
# Segunda pasada para destinos que aún no tienen hub
pt2 = (
    hubsdestination.loc[hubsdestination['near_hub'].isna(), ['lat', 'lng', 'city', 'state']]
    .drop_duplicates(subset=['lat', 'lng'])
)
print(f"Destinos sin hub tras primera pasada: {pt2.shape[0]}")

Destinos sin hub tras primera pasada: 0


In [42]:
hub_assign = pd.read_csv('../utils/hubs_assign.csv')
hubs_2_dest = hub_assignment(pt2, hub_assign, '../utils/hubs_id.xlsx')

mask = hubs_2_dest['state'].isna()
hubs_2_dest.loc[mask, 'state'] = hubs_2_dest.loc[mask, 'near_hub'].apply(extract_state)

hubs_2_dest.to_csv('../otros/hubsdestinations2.csv', index=False, float_format='%.10f')

Cantidad de ciudades inicial: 0
Cantidad de ciudades a procesar: 0
Cantidad de ciudades a procesar borrando duplicados: 0


,city,state_id,lat,lng


voy a procesar estas ciudades 0
se guardaron los cambios2


,lat,lng,city,state,new_name,hubs_50m,hubs_100m,hubs_150m


In [43]:
# Combinar origen-cache + pt1 + pt2 para cubrir todas las coordenadas de destino
hubs_dest_all = (
    pd.concat([
        hubs_origenes[['lat', 'lng', 'near_hub']],
        hubs_1[['lat', 'lng', 'near_hub']],
        hubs_2_dest[['lat', 'lng', 'near_hub']],
    ])
    .drop_duplicates(subset=['lat', 'lng'])
    .reset_index(drop=True)
)

hubsdestination = (
    hubsdestination
    .drop(columns='near_hub')
    .merge(hubs_dest_all[['lat', 'lng', 'near_hub']], on=['lat', 'lng'], how='left')
)

In [44]:
# Rellenar estado faltante en hubsdestination
mask = hubsdestination['state'].isna()
hubsdestination.loc[mask, 'state'] = hubsdestination.loc[mask, 'near_hub'].apply(extract_state)
hubsdestination

,lat,lng,city,state,ID,near_hub
0,40.686070,-74.322300,SPRINGFIELD TOWNSHIP,NY,494411154,New York (NY)
1,37.138401,-119.685300,O'NEALS,CA,495610380,Fresno (CA)
2,42.278702,-73.258598,SOUTH LEE,MA,491777519,Springfield (MA)
3,34.284401,-118.366890,LAKE VIEW TERRACE,CA,495598763,Los Angeles (CA)
4,32.344398,-89.327102,SCOTT,MS,495399133,Philadelphia (MS)
...,...,...,...,...,...,...
3790352,44.083890,-93.225830,Owatonna,MN,LLAGH0U4+ZS0Eue1x,Faribault (MN)
3790353,46.228610,-119.099440,Pasco,WA,LLACdHvD+ZS0EXdZ3,Yakima (WA)
3790354,33.435280,-112.357500,Goodyear,AZ,LLAKVsMF+DS3JexgN,Glendale (AZ)
3790355,33.952500,-84.550000,Marietta,GA,LLADhspU+ZS0EeGXM,Atlanta (GA)


In [45]:
hubs_1.to_csv('../otros/hubsdestinations.csv')

In [46]:
df = df.merge(
    hubsdestination[['ID', 'near_hub', 'state']].drop_duplicates(subset='ID'),
    on='ID',
    how='left'
)

In [47]:
mask_hub = (df['HubDestination'].isna()) & (df['LatDestination'].notna()) & (df['LngDestination'].notna())
df.loc[mask_hub, 'HubDestination'] = df.loc[mask_hub, 'near_hub']

In [48]:
mask_state = (df['StateDestination'].isna()) & (df['state'].notna())
df.loc[mask_state, 'StateDestination'] = df.loc[mask_state, 'state']
df = df.drop(['state', 'near_hub'], axis=1)

In [49]:
hubs_name = pd.read_excel('../utils/hubs_id.xlsx')
old_hubs = hubs_name['old_name']
new_hubs = hubs_name['new_name']
hubschange = set(old_hubs).difference(set(new_hubs))

df.loc[df['HubDestination'].isin(hubschange), 'HubDestination'] = df['HubDestination'].replace(city_pairs)
df.loc[~df['HubDestination'].isin(new_hubs), 'HubDestination'] = df['HubDestination'].replace(
    {'Manchester (MA, NH, ME)': 'Manchester (NH)'}
)

# Rellenar estado desde hub para cualquier fila que tenga hub pero no estado
mask = (df['StateDestination'].isna()) & (df['HubDestination'].notna())
df.loc[mask, 'StateDestination'] = df.loc[mask, 'HubDestination'].apply(extract_state)

In [50]:
# --- Verificación de destino ---
sin_hub_con_coords = df[(df['HubDestination'].isna()) & (df['LatDestination'].notna()) & (df['LngDestination'].notna())]
hub_sin_estado = df[(df['HubDestination'].notna()) & (df['StateDestination'].isna())]
print(f"Destinos con coords pero sin hub : {len(sin_hub_con_coords):,}")
print(f"Destinos con hub pero sin estado  : {len(hub_sin_estado):,}")

Destinos con coords pero sin hub : 0
Destinos con hub pero sin estado  : 0


In [51]:
df.shape

(57573489, 20)

In [52]:
usa_lat_min, usa_lat_max = 24.5, 49.5
usa_lon_min, usa_lon_max = -125, -66.5

def remove_out_of_bounds(df, lat_col, lng_col):
    mask = df[lat_col].notna() & df[lng_col].notna()
    in_bounds = (
        df[lat_col].between(usa_lat_min, usa_lat_max) &
        df[lng_col].between(usa_lon_min, usa_lon_max)
    )
    out = mask & ~in_bounds
    df.loc[out, lat_col] = None
    df.loc[out, lng_col] = None

# Corrección manual antes de limpiar
df.loc[df['CityOrigin'] == 'Owen', ['LatOrigin', 'LngOrigin']] = [38.458120, -85.54191]
df.loc[df['CityDestination'] == 'Owen', ['LatDestination', 'LngDestination']] = [38.458120, -85.54191]

remove_out_of_bounds(df, 'LatOrigin', 'LngOrigin')
remove_out_of_bounds(df, 'LatDestination', 'LngDestination')

In [53]:
df.shape

(57573489, 20)

In [54]:
from coord import process_row

def collect_invalid(df, lat_col, lng_col, city_col, state_col):
    mask = df[lat_col].isna() | (df[lat_col] == 0) | df[lng_col].isna() | (df[lng_col] == 0)
    return df.loc[mask, [city_col, state_col]].rename(columns={city_col: 'city', state_col: 'state'})

ciudades = pd.concat([
    collect_invalid(df, 'LatOrigin',      'LngOrigin',      'CityOrigin',      'StateOrigin'),
    collect_invalid(df, 'LatDestination', 'LngDestination', 'CityDestination', 'StateDestination'),
])
ciudades['city']  = ciudades['city'].str.lower()
ciudades['state'] = ciudades['state'].str.lower()
ciudades = ciudades.dropna(subset=['city']).drop_duplicates(subset=['city', 'state'])

In [55]:
from coord import _uscities_df

ciudades['city_key']  = ciudades['city'].str.lower().str.split(',').str[0].str.strip()
ciudades['state_key'] = ciudades['state'].str.lower().str.strip().fillna('')

con_estado = ciudades[ciudades['state_key'] != ''].merge(
    _uscities_df[['city_key', 'state_key', 'lat', 'lng']],
    on=['city_key', 'state_key'], how='left'
)
sin_estado = ciudades[ciudades['state_key'] == ''].merge(
    _uscities_df[['city_key', 'lat', 'lng']].drop_duplicates('city_key'),
    on='city_key', how='left'
)
ciudades = pd.concat([con_estado, sin_estado], ignore_index=True)
ciudades = ciudades.drop(columns=['city_key', 'state_key'])

In [56]:
# Si se activa geoapify para coordenadas faltantes:
# ciudades = ciudades.apply(process_row, axis=1)

In [57]:
def update_coords(df, lat_col, lng_col, city_col, state_col, ciudades):
    mask = df[lat_col].isna() | (df[lat_col] == 0) | df[lng_col].isna() | (df[lng_col] == 0)
    lookup = (
        df.loc[mask, ['ID', city_col, state_col]]
        .merge(
            ciudades[['city', 'state', 'lat', 'lng']],
            left_on=[city_col, state_col], right_on=['city', 'state'],
            how='left'
        )
        .set_index('ID')
    )
    ids = lookup.index
    df.loc[df['ID'].isin(ids), lat_col] = df.loc[df['ID'].isin(ids), 'ID'].map(lookup['lat'])
    df.loc[df['ID'].isin(ids), lng_col] = df.loc[df['ID'].isin(ids), 'ID'].map(lookup['lng'])

update_coords(df, 'LatOrigin',      'LngOrigin',      'CityOrigin',      'StateOrigin',      ciudades)
update_coords(df, 'LatDestination', 'LngDestination', 'CityDestination', 'StateDestination', ciudades)

In [58]:
df.to_parquet('parquetloadsfull.parquet')

In [59]:
df.shape

(57573489, 20)

In [60]:
df

,ID,Posted,PickUp,CityOrigin,LatOrigin,LngOrigin,CityDestination,LatDestination,LngDestination,Distance,Source,Equip,StateOrigin,HubOrigin,StateDestination,HubDestination,RatePerMile,Weight,Size,Broker_Shipper
0,0.0,2022-06-28 00:00:00,2022-07-01,Coquille,43.18020,-124.18410,Fremont,37.52650,-121.98520,530.0,DOFT,V,OR,Grants Pass (OR),CA,Stockton (CA),NaN,44000.0,NaN,NaN
1,1.0,2022-06-28 00:00:00,2022-06-30,Victorville,34.52770,-117.35360,Woodburn,45.14730,-122.86030,973.0,DOFT,V,CA,Los Angeles (CA),OR,Salem (OR),NaN,44000.0,NaN,NaN
2,2.0,2022-06-28 00:00:00,2022-06-28,Stockton,37.97650,-121.31090,Bend,44.05620,-121.30870,483.0,DOFT,V,CA,Stockton (CA),OR,Salem (OR),NaN,17000.0,NaN,NaN
3,3.0,2022-06-28 00:00:00,2022-07-01,Olivehurst,39.08170,-121.55490,Hermiston,45.83230,-119.28580,613.0,DOFT,V,CA,Stockton (CA),OR,Yakima (WA),NaN,27000.0,NaN,NaN
4,4.0,2022-06-28 00:00:00,2022-07-06,Moreno Valley,33.92440,-117.20450,Albany,44.62740,-123.09660,958.0,DOFT,V,CA,San Diego (CA),OR,Salem (OR),NaN,41000.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57573484,LLAGH0U4+ZS0Eue1x,2024-03-31 05:02:20,NaT,El Paso,31.75861,-106.48639,Owatonna,44.08389,-93.22583,1314.0,DAT,Van,TX,NaN,MN,Faribault (MN),1.712329,40648.0,53.0,ALLEN LUND COMPANY LLC
57573485,LLACdHvD+ZS0EXdZ3,2024-03-20 20:00:42,NaT,Portland,45.52361,-122.67500,Pasco,46.22861,-119.09944,212.0,DAT,Flatbed,OR,NaN,WA,Yakima (WA),3.301887,47200.0,48.0,TITANIUM AMERICAN LOGISTICS INC
57573486,LLAKVsMF+DS3JexgN,2024-04-08 09:36:02,NaT,Temecula,33.49361,-117.14750,Goodyear,33.43528,-112.35750,330.0,DAT,Reefer,CA,NaN,AZ,Glendale (AZ),2.878788,8239.0,53.0,MID AMERICA FREIGHT LOGISTICS LLC
57573487,LLADhspU+ZS0EeGXM,2024-03-23 18:22:19,NaT,McMinnville,35.68333,-85.77000,Marietta,33.95250,-84.55000,169.0,DAT,Van,TN,NaN,GA,Atlanta (GA),4.142071,44000.0,53.0,LANDSTAR RANGER
